# IMPLEMENTASI 1D LINE SEARCH
* Nama  : Alfian Daffa Baihaqi
* NIU   : 25/570509/PTK/16981
* Prodi : Magister Teknik Elektro

Project Optimisasi 
Find the approximation of the optimal solution of the following objective function:
$$f(x)=x^2-sin(x)$$
with initial interval of $[0,2]$ and until the interval reduce to $<=0.2$

In [169]:
import numpy as np

In [170]:
# Definisikan fungsi dan turunannya
def f(x):
    return x**2 - np.sin(x)

def df(x):
    return 2*x - np.cos(x)

def d2f(x):
    return 2 + np.sin(x)

#analytic solution
x_true = 0.4501836

## 1. Golden Section Method

In [171]:
def golden_section(f, a, b, final_range):
    p = (3 - np.sqrt(5)) / 2
    i = 0
    # Header
    print(f"{'Iter':<5}{'a':>10}{'b':>10}{'c':>10}{'d':>10}{'f(c)':>12}{'f(d)':>12}{'New Interval':>18}")
    print("-"*87)

    while (b - a) > final_range:
        c = a + p * (b - a)
        d = a + (1 - p) * (b - a)
        fc = f(c)
        fd = f(d)
        if fc < fd:
            new_a = a
            new_b = d
        else:
            new_a = c
            new_b = b
        new_interval = f"[{new_a:.4f}, {new_b:.4f}]"

        # print sebelum update
        print(f"{i:<5}{a:>10.4f}{b:>10.4f}{c:>10.4f}{d:>10.4f}{fc:>12.4f}{fd:>12.4f}{new_interval:>18}")
        # update interval
        a, b = new_a, new_b
        i += 1

    return (a + b) / 2

In [172]:
#Definisikan initial interval
a = 0
b = 2
final_range = 0.2
min_golden_sect = golden_section(f, a, b, final_range)
print("\nGolden Section Minimum =", min_golden_sect)
error_golden_sect = np.abs(min_golden_sect-x_true)
print("Error: ", error_golden_sect)

Iter          a         b         c         d        f(c)        f(d)      New Interval
---------------------------------------------------------------------------------------
0        0.0000    2.0000    0.7639    1.2361     -0.1082      0.5834  [0.0000, 1.2361]
1        0.0000    1.2361    0.4721    0.7639     -0.2319     -0.1082  [0.0000, 0.7639]
2        0.0000    0.7639    0.2918    0.4721     -0.2025     -0.2319  [0.2918, 0.7639]
3        0.2918    0.7639    0.4721    0.5836     -0.2319     -0.2104  [0.2918, 0.5836]
4        0.2918    0.5836    0.4033    0.4721     -0.2298     -0.2319  [0.4033, 0.5836]

Golden Section Minimum = 0.49342219125178766
Error:  0.04323859125178764


## 2. Fibonnaci Method

In [173]:
def find_N(eps, initial_range, final_range):
    target = (1 + 2*eps) / (final_range / initial_range)
    print("Target for Fibonacci sequence:", target)
    F = [1,1]
    while F[-1] < target:
        F.append(F[-1] + F[-2])

    N = len(F) - 2
    return N, np.array(F)

def fibonacci_method(f, a, b, eps, initial_range, final_range):
    N, F = find_N(eps, initial_range, final_range)
    print("Fibonacci sequence:", F)
    print("N =", N)
    # Header tabel
    print(f"\n{'Iter':<5}{'rho':>10}{'a1':>12}{'f(a1)':>12}{'b1':>12}{'f(b1)':>12}{'Interval':>20}")
    print("-"*83)
    # Iterasi sebanyak N kali
    for k in range(1, N+1):
        rho = 1 - F[N-k+1] / F[N-k+2]
        a1 = a + rho * (b - a)
        b1 = a + (1 - rho) * (b - a)
        fa = f(a1)
        fb = f(b1)
        if fa < fb:
            b = b1
        else:
            a = a1
        interval = f"[{a:.4f}, {b:.4f}]"
        print(f"{k:<5}{rho:>10.4f}{a1:>12.4f}{fa:>12.4f}{b1:>12.4f}{fb:>12.4f}{interval:>20}")

    x_opt = (a + b) / 2
    return x_opt

In [174]:
# Parameter
eps = 0.1 #asumsikan nilainya (tidak diberikan dalam soal)
a = 0
b = 2
final_range = 0.2
initial_range = b-a

min_fibonacci = fibonacci_method(f, a, b, eps, initial_range, final_range)
print("\nFibonacci Minimum =", min_fibonacci)
error_fibonacci = np.abs(min_fibonacci-x_true)
print("Error: ", error_fibonacci)


Target for Fibonacci sequence: 11.999999999999998
Fibonacci sequence: [ 1  1  2  3  5  8 13]
N = 5

Iter        rho          a1       f(a1)          b1       f(b1)            Interval
-----------------------------------------------------------------------------------
1        0.3846      0.7692     -0.1039      1.2308      0.5720    [0.0000, 1.2308]
2        0.3750      0.4615     -0.2323      0.7692     -0.1039    [0.0000, 0.7692]
3        0.4000      0.3077     -0.2082      0.4615     -0.2323    [0.3077, 0.7692]
4        0.3333      0.4615     -0.2323      0.6154     -0.1986    [0.3077, 0.6154]
5        0.5000      0.4615     -0.2323      0.4615     -0.2323    [0.4615, 0.6154]

Fibonacci Minimum = 0.5384615384615385
Error:  0.08827793846153853


## 3. Bisection Method

In [175]:
def bisection_method(a, b, final_range):
    iter_num = 1
    # Header tabel
    print(f"{'Iter':<5}{'a':>10}{'b':>10}{'mid':>10}{'df(mid)':>12}{'New Interval':>18}")
    print("-"*65)

    while (b - a) > final_range:
        c = (a + b) / 2
        dfc = df(c)
        # tentukan interval baru
        if dfc == 0:
            new_a, new_b = a, b
            break
        elif df(a) * dfc < 0:
            new_a, new_b = a, c
        else:
            new_a, new_b = c, b
        interval = f"[{new_a:.4f}, {new_b:.4f}]"
        # print sebelum update
        print(f"{iter_num:<5}{a:>10.4f}{b:>10.4f}{c:>10.4f}{dfc:>12.6f}{interval:>18}")
        # update interval
        a, b = new_a, new_b
        iter_num += 1

    x_opt = (a + b) / 2
    return x_opt

In [176]:
# Initial interval
a = 0
b = 2
final_range = 0.2

min_bisection = bisection_method(a, b, final_range)
print(f"\nBisection Minimum = ", min_bisection)
error_bisection = np.abs(min_bisection-x_true)
print("Error: ", error_bisection)

Iter          a         b       mid     df(mid)      New Interval
-----------------------------------------------------------------
1        0.0000    2.0000    1.0000    1.459698  [0.0000, 1.0000]
2        0.0000    1.0000    0.5000    0.122417  [0.0000, 0.5000]
3        0.0000    0.5000    0.2500   -0.468912  [0.2500, 0.5000]
4        0.2500    0.5000    0.3750   -0.180508  [0.3750, 0.5000]

Bisection Minimum =  0.4375
Error:  0.012683600000000017


## 4. Newton Method

In [177]:
def newton_method(df, d2f, x, final_range):
    error = np.inf
    iter_num = 1

    # Header tabel
    print(f"{'Iter':<5}{'x':>10}{'df(x)':>12}{'d2f(x)':>12}{'x_new':>12}{'Error':>12}")
    print("-"*63)

    while error > final_range:
        dfx = df(x)
        d2fx = d2f(x)

        if d2fx == 0:
            print("Turunan kedua nol!")
            break

        x_new = x - dfx / d2fx
        error = abs(x_new - x)

        print(f"{iter_num:<5}{x:>10.4f}{dfx:>12.6f}{d2fx:>12.6f}{x_new:>12.4f}{error:>12.6f}")

        x = x_new
        iter_num += 1

    return x

In [178]:
a = 0
b= 2
x0 = (b-a)/2
final_range = 0.2
min_newton = newton_method(df, d2f, x0, final_range)
print("\nNewton Minimum =", min_newton)
error_newton = np.abs(min_newton-x_true)
print("Error: ", error_newton)

Iter          x       df(x)      d2f(x)       x_new       Error
---------------------------------------------------------------
1        1.0000    1.459698    2.841471      0.4863    0.513712
2        0.4863    0.088502    2.467347      0.4504    0.035869

Newton Minimum = 0.45041860473199363
Error:  0.0002350047319936155


## 5. Secant Method

In [179]:
def secant_method(df, x0, x1, final_range):
    error = np.inf
    iter_num = 1

    # Header tabel
    print(f"{'Iter':<5}{'x0':>10}{'x1':>10}{'df(x0)':>12}{'df(x1)':>12}{'x_new':>12}{'Error':>12}")
    print("-"*73)

    while error > final_range:
        dfx0 = df(x0)
        dfx1 = df(x1)

        if dfx1 - dfx0 == 0:
            print("Function values at x0 and x1 are the same. Stopping iteration.")
            break

        x_new = x1 - dfx1 * (x1 - x0) / (dfx1 - dfx0)
        error = abs(x_new - x1)

        print(f"{iter_num:<5}{x0:>10.4f}{x1:>10.4f}{dfx0:>12.6f}{dfx1:>12.6f}{x_new:>12.4f}{error:>12.6f}")

        x0, x1 = x1, x_new
        iter_num += 1

    return x_new

In [180]:
a = 0
b = 2
final_range = 0.2
min_secant = secant_method(df, a, b, final_range)
print(f"\nSecant Minimum x = {min_secant:.4f}")
error_secant = np.abs(min_secant-x_true)
print("Error: ", error_secant)

Iter         x0        x1      df(x0)      df(x1)       x_new       Error
-------------------------------------------------------------------------
1        0.0000    2.0000   -1.000000    4.416147      0.3693    1.630734
2        2.0000    0.3693    4.416147   -0.194060      0.4379    0.068643

Secant Minimum x = 0.4379
Error:  0.012273997949861892


### Analisis
| Metode | Informasi yang Digunakan | Jumlah Iterasi | Perkiraan Titik Minimum (x) | Error | Karakteristik |
|------|------|------|------|------|------|
| Golden Section Search | Nilai fungsi f(x) | 5 | 0.4934 | 0.04323 | Stabil, tidak memerlukan turunan, konvergensi relatif lambat |
| Fibonacci Search | Nilai fungsi f(x) | 5 | 0.5385 | 0.08827 | Interval ditentukan oleh deret Fibonacci |
| Bisection Method | Turunan pertama | 4 | 0.4375 | 0.01268 | Konvergensi stabil dan cukup cepat |
| Newton Method | Turunan pertama dan kedua | 2 | 0.4504 | 0.000235 | Konvergensi sangat cepat (kuadratik) |
| Secant Method | Turunan pertama | 2 | 0.4379 | 0.01227 | Tidak memerlukan turunan kedua, konvergensi cepat |

Solusi analitis nilai minimum dari persamaan $f(x)=x^2-sin(x)$ adalah $x=0.4501836$. Hasil perhitungan menunjukkan bahwa seluruh metode berhasil mempersempit daerah solusi di sekitar $𝑥≈0,45$, yang merupakan lokasi minimum fungsi. Metode berbasis pencarian interval, seperti Golden Section dan Fibonacci, bekerja dengan mengevaluasi nilai fungsi tanpa memerlukan turunan sehingga bersifat stabil namun konvergensinya relatif lebih lambat. Sebaliknya, metode yang memanfaatkan informasi turunan, seperti Bisection, Newton, dan Secant, menunjukkan kecepatan konvergensi yang lebih tinggi. Di antara metode yang diuji, Newton Method memberikan hasil paling efisien karena mampu mendekati solusi optimum dalam jumlah iterasi yang paling sedikit dengan memanfaatkan turunan pertama dan kedua dari fungsi.